# Complexity Heuristics Exploration

This notebook implements the heuristics-based routing approach outlined in `complexity-heuristics.md`.

Extract features from a query and decide whether to route it to a **Strong Model** or a **Weak Model** without using any real model inference.

In [1]:
import re
import pandas as pd

In [2]:
def extract_features(query):
    # 1. Token count (approximate as length/4 or just split)
    token_count = len(query.split())
    
    # 2. Word count
    word_count = len(re.findall(r'\w+', query))
    
    # 3. Complexity Keywords
    complexity_keywords = ["explain", "debug", "analyze", "compare", "why", "how", "diff", "refactor"]
    keyword_count = sum(1 for kw in complexity_keywords if kw in query.lower())
    
    # 4. Named Entity Count (Basic heuristic: Capitalized words not at start of sentence)
    # This is a crude proxy for named entities
    entities = re.findall(r'\b[A-Z][a-z]+\b', query)
    # Filter out start of sentence if possible, but for short queries this is fine
    entity_count = len(entities)
    
    # 5. Question marks / Multi-part questions
    question_count = query.count('?')
    multi_part = 1 if question_count > 1 or ';' in query or '\n' in query else 0
    
    # 6. Technical Terms / Code Detection
    tech_terms = ["function", "class", "def", "const", "api", "json", "sql", "import", "null", "undefined", "{ ", "}"]
    tech_score = sum(2 for term in tech_terms if term in query.lower())
    
    return {
        "tokens": token_count,
        "words": word_count,
        "keywords": keyword_count,
        "entities": entity_count,
        "questions": question_count,
        "multi_part": multi_part,
        "tech_score": tech_score
    }

## Sample Queries
Let's test with a mix of simple and complex queries.

In [3]:
samples = [
    "Hi there!",
    "What time is it in New York?",
    "Tell me a joke.",
    "Explain the difference between TCP and UDP in detail.",
    "Debug this Python code: def hello(): print('world'",
    "Analyze the fiscal policy of the US in 2023 vs 2024.",
    "How do I implement a binary search tree in C++?",
    "Can you compare React and Vue for a small scale application?"
]

results = []
for q in samples:
    features = extract_features(q)
    features['query'] = q
    results.append(features)

df = pd.DataFrame(results)
df = df[['query', 'tokens', 'words', 'keywords', 'entities', 'questions', 'multi_part', 'tech_score']]
df

,query,tokens,words,keywords,entities,questions,multi_part,tech_score
0,Hi there!,2,2,0,1,0,0,0
1,What time is it in New York?,7,7,0,3,1,0,0
2,Tell me a joke.,4,4,0,1,0,0,0
3,Explain the difference between TCP and UDP in ...,9,9,2,1,0,0,0
4,Debug this Python code: def hello(): print('wo...,7,8,1,2,0,0,2
5,Analyze the fiscal policy of the US in 2023 vs...,11,11,1,1,0,0,0
6,How do I implement a binary search tree in C++?,10,10,1,1,1,0,0
7,Can you compare React and Vue for a small scal...,11,11,1,3,1,0,0


## Defining the Router
Now we create a heuristic-based selection logic.

In [4]:
def route_query(query):
    feats = extract_features(query)
    
    # Heuristic score calculation
    # We can weight the features
    score = (feats['tokens'] * 0.1) + \
            (feats['keywords'] * 1.5) + \
            (feats['questions'] * 0.5) + \
            (feats['tech_score'] * 1.0) + \
            (feats['multi_part'] * 1.0)
    
    # Threshold for "Strong Model"
    threshold = 3.0
    
    if score >= threshold:
        return "Strong Model (GPT-4o)"
    else:
        return "Weak Model (Llama-3-8B)"

# Apply routing to samples
df['selection'] = df['query'].apply(route_query)
df[['query', 'selection']]

,query,selection
0,Hi there!,Weak Model (Llama-3-8B)
1,What time is it in New York?,Weak Model (Llama-3-8B)
2,Tell me a joke.,Weak Model (Llama-3-8B)
3,Explain the difference between TCP and UDP in ...,Strong Model (GPT-4o)
4,Debug this Python code: def hello(): print('wo...,Strong Model (GPT-4o)
5,Analyze the fiscal policy of the US in 2023 vs...,Weak Model (Llama-3-8B)
6,How do I implement a binary search tree in C++?,Strong Model (GPT-4o)
7,Can you compare React and Vue for a small scal...,Strong Model (GPT-4o)
